# Setup

In [1]:
import pandas as pd
import numpy as np
import cobra
import pickle
import matplotlib.pyplot as plt

import mygene
mg = mygene.MyGeneInfo()

plt.rc('font', size=20)

/Users/huahualiu/anaconda3/envs/huahualiu/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Read the transcriptomics

In [2]:
Cell = pd.read_excel('Data/GSM8561764_CMO301_AVERAGE.xlsx')
df_data =Cell.copy()
df_data=df_data.set_index('Genes')

print('Imported dataframe size:',len(df_data))
df_data.head()

Imported dataframe size: 24072


,Average
Genes,
AL627309.1,0.001946
AL627309.4,0.000000
AL669831.2,0.000000
AL669831.5,0.075912
FAM87B,0.000487


# Get the list of metabolic genes from Recon3D

In [3]:
model = cobra.io.load_matlab_model('Models/Recon3DModel_301.mat')

l_genes_full = [g.id for g in model.genes]
l_genes = [g.id.split('.')[0] for g in model.genes]

INFO:cobra.core.model:The current solver interface glpk doesn't support setting the optimality tolerance.


# Set up the dataframe

In [4]:
df = pd.DataFrame(index=l_genes,columns=['Average'] + ['Symbol', 'Alias', 'Name'])
df

,Average,Symbol,Alias,Name
8639,NaN,NaN,NaN,NaN
26,NaN,NaN,NaN,NaN
314,NaN,NaN,NaN,NaN
314,NaN,NaN,NaN,NaN
1591,NaN,NaN,NaN,NaN
...,...,...,...,...
170712,NaN,NaN,NaN,NaN
1346,NaN,NaN,NaN,NaN
1340,NaN,NaN,NaN,NaN
84701,NaN,NaN,NaN,NaN


# Query from entrezgene to gene symbols, alias and name
This query yields 3 genes that are not found: 0 does not exist, the other two missing genes have been withdrawn from NCBI.

The warning about duplicates is due to the fact that the input entrez gene list contains duplicates.

In [5]:
out = mg.querymany(l_genes, scopes='entrezgene', fields='symbol,alias,name', species='human')

df = pd.DataFrame(out)[['query','symbol','alias','name']]
df.columns = ['Entrez','Symbol','Alias','Name']
df = df.set_index('Entrez')

columns=['Average']
df_nan = pd.DataFrame(index=df.index, columns=columns)
df = pd.concat([df, df_nan], axis=1)
# reset index to full gene IDs
df.index = l_genes_full

df.head()

INFO:biothings.client:querying 1-1000...
INFO:biothings.client:done.
INFO:biothings.client:querying 1001-2000...
INFO:biothings.client:done.
INFO:biothings.client:querying 2001-2248...
INFO:biothings.client:done.
INFO:biothings.client:Finished.
INFO:biothings.client:Pass "returnall=True" to return complete lists of duplicate or missing query terms.


,Symbol,Alias,Name,Average
8639.1,AOC3,"[HPAO, SSAO, VAP-1, VAP1]",amine oxidase copper containing 3,NaN
26.1,AOC1,"[ABP, ABP1, DAO, DAO1, KAO, KDAO]",amine oxidase copper containing 1,NaN
314.2,AOC2,"[DAO2, RAO, SSAO]",amine oxidase copper containing 2,NaN
314.1,AOC2,"[DAO2, RAO, SSAO]",amine oxidase copper containing 2,NaN
1591.1,CYP24A1,"[CP24, CYP24, HCAI, HCINF1, P450-CC24]",cytochrome P450 family 24 subfamily A member 1,NaN


# Match the symbol column to the dataset and save the TPM scores

In [6]:
count = 0
columns=['Average']
df_data_index_list = df_data.index.tolist()  # Convert df_data.index to a list
for indx, row in df.iterrows():
    s = df.at[indx, 'Symbol']
    
    if s in df_data_index_list:
        v = df_data.loc[s, columns]
        df.loc[indx, columns] = v.values
    else:
        count += 1

print('Could not find', count, 'gene symbols in the dataset')
df.head()

Could not find 191 gene symbols in the dataset


,Symbol,Alias,Name,Average
8639.1,AOC3,"[HPAO, SSAO, VAP-1, VAP1]",amine oxidase copper containing 3,0.004866
26.1,AOC1,"[ABP, ABP1, DAO, DAO1, KAO, KDAO]",amine oxidase copper containing 1,0.000487
314.2,AOC2,"[DAO2, RAO, SSAO]",amine oxidase copper containing 2,0.001946
314.1,AOC2,"[DAO2, RAO, SSAO]",amine oxidase copper containing 2,0.001946
1591.1,CYP24A1,"[CP24, CYP24, HCAI, HCINF1, P450-CC24]",cytochrome P450 family 24 subfamily A member 1,NaN


In [7]:
exclude_columns = ['Symbol', 'Alias', 'Name']
columns_to_check = df.columns.difference(exclude_columns)
missing = df[df[columns_to_check].isnull().any(axis=1)].index.tolist()
columns=['Average']

for indx in missing:
    aliases = df.at[indx,'Alias']
    
    if type(aliases) == float:
        continue
    
    for alias in aliases:
        if alias in df_data.index:
            v = df_data.loc[s, columns]
            df.loc[indx, columns] = v.values
            break
            
missing = df[df[columns_to_check].isnull().any(axis=1)].index.tolist()
print('After alias searching we are left with', len(missing),'remaining missing genes.')

# df.loc[missing][['Symbol','Alias','Name']].to_excel('Tables/Patient_data/Unmappable_RNA-seq_genes_Patient.xlsx')

df.loc[missing][['Symbol','Alias','Name']]

After alias searching we are left with 159 remaining missing genes.


,Symbol,Alias,Name
1591.1,CYP24A1,"[CP24, CYP24, HCAI, HCINF1, P450-CC24]",cytochrome P450 family 24 subfamily A member 1
130.1,ADH6,ADH-5,alcohol dehydrogenase 6 (class V)
127.1,ADH4,"[ADH-2, HEL-S-4]","alcohol dehydrogenase 4 (class II), pi polypep..."
124.1,ADH1A,ADH1,"alcohol dehydrogenase 1A (class I), alpha poly..."
131.1,ADH7,NaN,"alcohol dehydrogenase 7 (class IV), mu or sigm..."
...,...,...,...
251.1,ALPG,"[ALPPL, ALPPL2, GCAP]","alkaline phosphatase, germ cell"
10941.1,UGT2A1,UDPGT2A1,UDP glucuronosyltransferase family 2 member A1...
79799.1,UGT2A3,NaN,UDP glucuronosyltransferase family 2 member A3
0,NaN,NaN,NaN


# Assign the score to the missing genes

## assign the maximum score to the missing genes

In [8]:
columns=['Average']
df.loc[missing,columns] = 1

df.loc[missing]

,Symbol,Alias,Name,Average
1591.1,CYP24A1,"[CP24, CYP24, HCAI, HCINF1, P450-CC24]",cytochrome P450 family 24 subfamily A member 1,1
130.1,ADH6,ADH-5,alcohol dehydrogenase 6 (class V),1
127.1,ADH4,"[ADH-2, HEL-S-4]","alcohol dehydrogenase 4 (class II), pi polypep...",1
124.1,ADH1A,ADH1,"alcohol dehydrogenase 1A (class I), alpha poly...",1
131.1,ADH7,NaN,"alcohol dehydrogenase 7 (class IV), mu or sigm...",1
...,...,...,...,...
251.1,ALPG,"[ALPPL, ALPPL2, GCAP]","alkaline phosphatase, germ cell",1
10941.1,UGT2A1,UDPGT2A1,UDP glucuronosyltransferase family 2 member A1...,1
79799.1,UGT2A3,NaN,UDP glucuronosyltransferase family 2 member A3,1
0,NaN,NaN,NaN,1


In [9]:
df.to_csv('Data/RNAseq_data_with_entrez_genes_Cell_AVERAGE(missing=1).txt', sep="\t", index_label='gene')

In [10]:
df.to_csv('Data/RNAseq_data_with_entrez_genes_Cell_AVERAGE(missing=1).csv', sep="\t", index_label='gene')

## assign the minimum score to the missing genes

In [11]:
columns=['Average']
df.loc[missing,columns] = 0

df.loc[missing]

,Symbol,Alias,Name,Average
1591.1,CYP24A1,"[CP24, CYP24, HCAI, HCINF1, P450-CC24]",cytochrome P450 family 24 subfamily A member 1,0
130.1,ADH6,ADH-5,alcohol dehydrogenase 6 (class V),0
127.1,ADH4,"[ADH-2, HEL-S-4]","alcohol dehydrogenase 4 (class II), pi polypep...",0
124.1,ADH1A,ADH1,"alcohol dehydrogenase 1A (class I), alpha poly...",0
131.1,ADH7,NaN,"alcohol dehydrogenase 7 (class IV), mu or sigm...",0
...,...,...,...,...
251.1,ALPG,"[ALPPL, ALPPL2, GCAP]","alkaline phosphatase, germ cell",0
10941.1,UGT2A1,UDPGT2A1,UDP glucuronosyltransferase family 2 member A1...,0
79799.1,UGT2A3,NaN,UDP glucuronosyltransferase family 2 member A3,0
0,NaN,NaN,NaN,0


In [12]:
df.to_csv('Data/RNAseq_data_with_entrez_genes_Cell_AVERAGE(missing=0).csv', sep="\t", index_label='gene')

In [13]:
df.to_csv('Data/RNAseq_data_with_entrez_genes_Cell_AVERAGE(missing=0).txt', sep="\t", index_label='gene')